# Aprendizado de Máquina — Lista prática 10

## Máquinas de Vetores de Suporte (SVM)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A SVM tem dois hiperparâmetros que interagem — $C$ e $\gamma$ — e uma grade
bidimensional é a primeira do curso. O resultado dela, neste banco, é um daqueles
que valem mais do que a resposta certa:

> **o melhor RBF encontrado pela busca é o que mais se parece com um kernel
> linear. E o kernel linear, sozinho, chega quase lá — em uma fração do tempo.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import time

import numpy as np
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — o que $C$ faz com a margem

Duas nuvens gaussianas que se sobrepõem um pouco — não são linearmente
separáveis, então a margem rígida do Exercício 1 da Lista Teórica 10 nem
existiria aqui.

Ajuste a SVM linear com três valores de $C$ e observe **três** quantidades: o
número de vetores de suporte, a largura da margem $1/\lVert w\rVert$, e a
acurácia de treino.

In [ ]:
rng = np.random.default_rng(2026)
n = 60

X = np.vstack([rng.normal([-1.5, 0], 0.8, size=(n // 2, 2)),
               rng.normal([1.5, 0], 0.8, size=(n // 2, 2))])
y = np.r_[-np.ones(n // 2), np.ones(n // 2)]

print("        C     vetores de suporte    margem   acuracia de treino")
for c in (0.01, 1.0, 100.0):
    modelo = SVC(kernel="linear", C=c).fit(X, y)               # (a) e (b)
    margem = 1 / np.linalg.norm(modelo.coef_[0])                 # (c)
    print(f"  {c:7.2f}          {modelo.n_support_.sum():2d}/{n}        "
          f"{margem:.4f}        {modelo.score(X, y):.4f}")

Deve imprimir:

```
        C     vetores de suporte    margem   acuracia de treino
     0.01          44/60        2.0844        0.9500
     1.00          10/60        0.5720        0.9833
   100.00           5/60        0.2913        0.9667
```

As duas primeiras colunas confirmam o Exercício 2 da Lista Teórica 10, na
direção prevista:

- **$C$ pequeno** ($0{,}01$): margem larga ($2{,}08$) e **44 dos 60** pontos
  dentro dela, todos vetores de suporte. É o regime regularizado — a fronteira é
  determinada por quase todo o conjunto, e por isso é estável.
- **$C$ grande** ($100$): margem estreita ($0{,}29$) e apenas **5** vetores de
  suporte. A fronteira passa a depender de cinco observações, o que é flexível e
  instável ao mesmo tempo.

A terceira coluna traz uma surpresa: a acurácia de treino **não é monótona** em
$C$ — sobe de $0{,}9500$ para $0{,}9833$ e depois cai para $0{,}9667$. Isso
parece contradizer "mais flexível ajusta melhor o treino", mas não contradiz: a
SVM **não minimiza o erro de classificação**, e sim a perda *hinge* mais a
penalização (Exercício 4 da Lista Teórica 10). Nada garante que aumentar $C$
melhore o erro $0$--$1$ no treino; garante apenas que se tolere menos violação
da margem.

> **Sua vez.** Desenhe as duas nuvens e, por cima, a reta de decisão e as duas
> retas da margem, para $C=0{,}01$ e $C=100$. Marque os vetores de suporte com um
> círculo. (`modelo.support_vectors_` devolve as coordenadas.)

---
## Exercício 2 — a grade bidimensional

No kernel RBF, $K(x,z) = \exp(-\gamma\lVert x-z\rVert^2)$, os dois
hiperparâmetros fazem coisas diferentes: $C$ controla quanto se tolera errar, e
$\gamma$ controla o **alcance** de cada observação. Eles interagem, então é
preciso buscá-los juntos.

Padronizar antes é obrigatório: $\gamma$ multiplica uma distância euclidiana, e
sem padronizar a covariável de maior escala domina a soma.

In [ ]:
dados = load_breast_cancer()
X_bc, y_bc = dados.data, dados.target

tubo = Pipeline([("escala", StandardScaler()), ("svm", SVC(kernel="rbf"))])

grade = {
    "svm__C":     np.logspace(-2, 3, 6),                         # (a) de 0,01 a 1000
    "svm__gamma": np.logspace(-4, 1, 6),                         # (b) de 0,0001 a 10
}

cv = skm.StratifiedKFold(5, shuffle=True, random_state=2026)
busca = skm.GridSearchCV(tubo, grade, cv=cv, scoring="accuracy", n_jobs=-1).fit(X_bc, y_bc)

print(f"melhor: C = {busca.best_params_['svm__C']:g}, "
      f"gamma = {busca.best_params_['svm__gamma']:g}")
print(f"acuracia (CV) = {busca.best_score_:.4f}")

In [ ]:
tabela = busca.cv_results_["mean_test_score"].reshape(6, 6)     # (a) e (b) C x gamma

print("linhas = C, colunas = gamma")
print("        " + "".join(f"{g:9.4g}" for g in grade["svm__gamma"]))
for c, linha in zip(grade["svm__C"], tabela):
    print(f"  {c:7.4g} " + "".join(f"{v:9.4f}" for v in linha))

Deve imprimir `melhor: C = 1000, gamma = 0.0001`, `acuracia (CV) = 0.9772`, e a
tabela:

```
linhas = C, colunas = gamma
           0.0001    0.001     0.01      0.1        1       10
     0.01    0.6274   0.6274   0.6274   0.6274   0.6274   0.6274
      0.1    0.6274   0.7908   0.9490   0.9403   0.6274   0.6274
        1    0.7925   0.9490   0.9649   0.9631   0.6309   0.6274
       10    0.9508   0.9737   0.9754   0.9596   0.6345   0.6274
      100    0.9754   0.9701   0.9649   0.9578   0.6345   0.6274
     1000    0.9772   0.9666   0.9649   0.9578   0.6345   0.6274
```

Três coisas para ler nesta tabela, e nenhuma delas é o número $0{,}9772$.

**As colunas da direita são todas $0{,}6274$**, que é exatamente a prevalência da
classe majoritária. Com $\gamma \ge 1$ o kernel decai tão rápido que cada
observação só "enxerga" a si mesma; o modelo memoriza o treino e, em qualquer
ponto novo, não tem vizinho nenhum a que recorrer — devolve a classe majoritária.
É o análogo do KNN com $k=1$ levado ao extremo.

**A primeira linha também é constante.** Com $C=0{,}01$ o custo de errar é tão
baixo que a solução é $w \approx 0$, e o classificador degenera do outro
lado. Os dois extremos falham, com a mesma acurácia, por motivos opostos.

**O ótimo está no canto da grade** ($C=1000$, $\gamma=0{,}0001$), o que é sempre
um sinal de alerta: a busca pode estar querendo sair pela borda. E há uma
interpretação: $\gamma\to 0$ faz o kernel RBF tender a um **kernel linear**, e $C$
grande compensa a penalização. Ou seja, a melhor SVM com kernel RBF encontrada
aqui é a que mais se parece com uma SVM linear.

Vale conferir essa última leitura diretamente.

In [ ]:
linear = Pipeline([("escala", StandardScaler()), ("svm", SVC(kernel="linear"))])   # (a)
acc_linear = skm.cross_val_score(linear, X_bc, y_bc, cv=cv).mean()

print(f"RBF, melhor da grade : {busca.best_score_:.4f}")
print(f"kernel linear (C=1)  : {acc_linear:.4f}")

Deve imprimir `RBF, melhor da grade : 0.9772` e `kernel linear (C=1) : 0.9719`.

A diferença é de **0,5 ponto percentual** — e o kernel linear chegou lá com um
hiperparâmetro no valor padrão, contra 36 ajustes da busca.

A conclusão prática é a que o curso vem repetindo: **comece pelo modelo simples e
só suba a complexidade se ela pagar.** Aqui o `breast_cancer` é essencialmente
linearmente separável depois de padronizado, e a flexibilidade do RBF não tem o
que fazer. Compare com a Aula 08, em que a logística (fronteira linear) também
tinha ganhado de LDA, QDA e naive Bayes neste mesmo banco: são três evidências
independentes de que a fronteira aqui é linear.

---
## Exercício 3 — o custo, medido

A SVM com kernel resolve um problema quadrático cuja matriz de Gram é
$n\times n$: a teoria promete custo entre $O(n^2)$ e $O(n^3)$. Já o `LinearSVC`
resolve o problema **primal**, com custo linear em $n$.

Meça os dois, e estime o expoente por regressão nos logaritmos — mesma técnica
do Exercício 3 da Lista prática 05.

In [ ]:
ns = np.array([1000, 2000, 4000, 8000, 16000])
tempo_svc, tempo_lin = [], []

for tamanho in ns:
    Xg, yg = make_classification(n_samples=int(tamanho), n_features=20,
                                 n_informative=10, random_state=0)
    t0 = time.time()
    SVC(kernel="rbf").fit(Xg, yg)
    tempo_svc.append(time.time() - t0)                           # (a)

    t0 = time.time()
    LinearSVC(max_iter=5000, dual=True).fit(Xg, yg)
    tempo_lin.append(time.time() - t0)

tempo_svc, tempo_lin = np.array(tempo_svc), np.array(tempo_lin)

print("      n      SVC(rbf)   LinearSVC")
for tamanho, a, b in zip(ns, tempo_svc, tempo_lin):
    print(f"  {tamanho:6d}   {a:8.3f}s  {b:8.3f}s")

print(f"\nexpoente SVC       : {np.polyfit(np.log(ns), np.log(tempo_svc), 1)[0]:+.3f}")   # (b) e (c)
print(f"expoente LinearSVC : {np.polyfit(np.log(ns), np.log(tempo_lin), 1)[0]:+.3f}")

Os tempos dependem da máquina, mas a **forma** não. Na máquina em que o gabarito
foi gerado:

```
      n      SVC(rbf)   LinearSVC
    1000      0.020s     0.289s
    2000      0.054s     0.569s
    4000      0.156s     1.166s
    8000      0.654s     2.645s
   16000      1.461s     5.552s

expoente SVC       : +1.598
expoente LinearSVC : +1.074
```

Os expoentes confirmam a teoria: o `SVC` cresce claramente **mais rápido que
linear** ($+1{,}60$; a cota é $O(n^2)$ a $O(n^3)$, e a implementação usa cache e
heurísticas que a mantêm abaixo do pior caso), e o `LinearSVC` é
**essencialmente linear** ($+1{,}07$).

Mas repare no que a coluna de tempos diz e a coluna de expoentes esconde: **em
todos os $n$ testados, o `LinearSVC` é mais lento**, de duas a quatro vezes. A
vantagem assintótica ainda não pagou a diferença de constantes.

Dá para estimar quando ela paga. Se $T_{\text{SVC}} \propto n^{1{,}60}$ e
$T_{\text{lin}} \propto n^{1{,}07}$, a razão entre eles cai como $n^{-0{,}52}$;
para fechar o fator de $3{,}8$ que existe em $n=16\,000$, seria preciso
$(n/16\,000)^{0{,}52} = 3{,}8$, isto é $n \approx 200\,000$.

É a moral do exercício, e ela vale para qualquer análise de complexidade:
**expoente é sobre o comportamento no limite, não sobre o seu problema.** Para
$n$ na casa dos milhares, use o `SVC`; a partir de algumas centenas de milhares,
o `LinearSVC` — ou os métodos lineares da Aula 02, que escalam melhor que os
dois.

> **Sua vez.** Acrescente $n = 32\,000$ à lista e refaça o ajuste do expoente. Ele
> sobe ou desce? O que isso sugere sobre o regime em que a implementação está?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | de $C=0{,}01$ a $C=100$: vetores de suporte caem de 44 para 5, margem de 2,08 para 0,29 |
| 1 | a acurácia de treino **não** é monótona em $C$ — a SVM não minimiza o erro $0$--$1$ |
| 2 | com $\gamma\ge 1$ a acurácia trava em 0,6274, que é a prevalência da classe majoritária |
| 2 | o melhor RBF ($\gamma=0{,}0001$) é o que mais se parece com um linear — e o linear sozinho dá 0,9719 |
| 3 | expoentes medidos: `SVC` $+1{,}60$, `LinearSVC` $+1{,}07$ |
| 3 | ainda assim o `LinearSVC` é **mais lento** em todos os $n$ testados; só compensa perto de $n=200\,000$ |

**A seguir.** A Aula 11 fecha o bloco de classificação trazendo o KNN e as
árvores para cá, e põe todas as fronteiras do bloco lado a lado nos mesmos dados.